In [25]:
import os
import sys
import pandas as pd
import re
from collections import Counter

# 환경 설정
project_dir = "/data/ephemeral/home/nlp-5/eunbyul/joe"
sys.path.append(project_dir)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 데이터 로드
train_df = pd.read_csv(os.path.join(project_dir, 'data', 'train.csv'))
val_df = pd.read_csv(os.path.join(project_dir, 'data', 'dev.csv'))
test_df = pd.read_csv(os.path.join(project_dir, 'data', 'test.csv'))

# 전체를 하나의 dict로 관리
datasets = {'train': train_df, 'dev': val_df, 'test': test_df}

----

마스킹 토큰 누락/불일치

In [3]:
# 마스킹 토큰 종류 정의 (필요시 추가)
MASK_TOKENS = [
    '#Person1#', '#Person2#', '#Person3#', '#Person4#', '#Person5#', '#Person6#', '#Person7#',
    '#PhoneNumber#', '#Address#', '#DateOfBirth#', '#PassportNumber#',
    '#SSN#', '#CardNumber#', '#CarNumber#', '#Email#'
]

def count_masking_tokens(text, tokens=MASK_TOKENS):
    counts = {}
    for t in tokens:
        counts[t] = len(re.findall(re.escape(t), str(text)))
    return counts

def eda_masking_discrepancy(df):
    dialogue_mask = df['dialogue'].apply(lambda x: count_masking_tokens(x))
    summary_mask = df['summary'].apply(lambda x: count_masking_tokens(x))

    mask_df = pd.DataFrame({
        'dialogue': dialogue_mask,
        'summary': summary_mask
    })

    # dialogue와 summary의 토큰별 count 차이 집계
    discrepancy = {}
    for token in MASK_TOKENS:
        diff_count = (mask_df['dialogue'].apply(lambda d: d[token]) != mask_df['summary'].apply(lambda s: s[token])).sum()
        discrepancy[token] = diff_count

    print("=== [MASKING TOKEN DISCREPANCY] ===")
    for token, diff in discrepancy.items():
        print(f"{token}: {diff} samples에서 불일치")
    print("총 샘플 수:", len(df))
    return discrepancy


In [9]:
print(eda_masking_discrepancy(train_df))
print(eda_masking_discrepancy(val_df))

=== [MASKING TOKEN DISCREPANCY] ===
#Person1#: 12207 samples에서 불일치
#Person2#: 11941 samples에서 불일치
#Person3#: 109 samples에서 불일치
#Person4#: 14 samples에서 불일치
#Person5#: 5 samples에서 불일치
#Person6#: 2 samples에서 불일치
#Person7#: 1 samples에서 불일치
#PhoneNumber#: 150 samples에서 불일치
#Address#: 77 samples에서 불일치
#DateOfBirth#: 20 samples에서 불일치
#PassportNumber#: 5 samples에서 불일치
#SSN#: 3 samples에서 불일치
#CardNumber#: 12 samples에서 불일치
#CarNumber#: 7 samples에서 불일치
#Email#: 13 samples에서 불일치
총 샘플 수: 12457
{'#Person1#': np.int64(12207), '#Person2#': np.int64(11941), '#Person3#': np.int64(109), '#Person4#': np.int64(14), '#Person5#': np.int64(5), '#Person6#': np.int64(2), '#Person7#': np.int64(1), '#PhoneNumber#': np.int64(150), '#Address#': np.int64(77), '#DateOfBirth#': np.int64(20), '#PassportNumber#': np.int64(5), '#SSN#': np.int64(3), '#CardNumber#': np.int64(12), '#CarNumber#': np.int64(7), '#Email#': np.int64(13)}
=== [MASKING TOKEN DISCREPANCY] ===
#Person1#: 484 samples에서 불일치
#Person2#: 480 samples에서 불일

지시표현/지시어 패턴 빈도

In [30]:
# 지시어 후보 패턴 (필요시 확장)
DEICTIC_PHRASES = [
    '그 사람', '이 사람', '그녀', '그', '이 분', '저분', '그 팀', '그곳', '그날',
    '그거', '이거', '그건', '이건', '거기', '저기', '여기', '그쪽', '저 사람', '그분'
]

def eda_deictic_phrases(df, text_col='dialogue'):
    pattern = '|'.join([re.escape(word) + r'([가-힣]*)' for word in DEICTIC_PHRASES]) # 조사결합형까지 탐지
    results = df[text_col].apply(lambda x: re.findall(pattern, str(x)))
    flat = [item for sublist in results for item in sublist if item]
    from collections import Counter
    counts = Counter(flat)
    print(f"=== [DEICTIC PHRASE STATS in {text_col}] ===")
    for phrase, freq in counts.most_common(20):
        print(f"{phrase}: {freq}회")
    return counts


In [31]:
eda_deictic_phrases(train_df, 'dialogue')
eda_deictic_phrases(train_df, 'summary')

=== [DEICTIC PHRASE STATS in dialogue] ===
('', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 8605회
('', '', '', '럼', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 3570회
('', '', '', '리고', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 2406회
('', '', '', '래', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 2066회
('', '', '', '렇게', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 1869회
('', '', '', '냥', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 1858회
('', '', '', '런데', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 1814회
('', '', '', '래서', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 1559회
('', '', '', '게', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 1491회
('', '', '', '런', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 1458회
('', '', '', '건', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''): 1200회
('', '', '', '거', '

Counter({('',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          ''): 564,
         ('',
          '',
          '',
          '들은',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          ''): 433,
         ('',
          '',
          '',
          '의',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          ''): 216,
         ('',
          '',
          '의',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          '',
          ''

불용어/불필요 문장 빈도

In [11]:
STOPWORDS = ['안녕하세요', '감사합니다', '네, 알겠습니다', '네', '아니요', '수고하세요', '잘 부탁드립니다', '고맙습니다', '네네', '예']

def eda_stopwords(df, text_col='dialogue', stopwords=STOPWORDS):
    found = {w: 0 for w in stopwords}
    for line in df[text_col]:
        for w in stopwords:
            found[w] += str(line).count(w)
    print(f"=== [STOPWORD STATS in {text_col}] ===")
    for w, cnt in found.items():
        print(f"{w}: {cnt}회")
    return found


In [12]:
eda_stopwords(train_df, 'dialogue')
eda_stopwords(train_df, 'summary')

=== [STOPWORD STATS in dialogue] ===
안녕하세요: 2090회
감사합니다: 2431회
네, 알겠습니다: 203회
네: 18303회
아니요: 1103회
수고하세요: 0회
잘 부탁드립니다: 2회
고맙습니다: 123회
네네: 1회
예: 6251회
=== [STOPWORD STATS in summary] ===
안녕하세요: 0회
감사합니다: 1회
네, 알겠습니다: 0회
네: 142회
아니요: 0회
수고하세요: 0회
잘 부탁드립니다: 0회
고맙습니다: 0회
네네: 0회
예: 1224회


{'안녕하세요': 0,
 '감사합니다': 1,
 '네, 알겠습니다': 0,
 '네': 142,
 '아니요': 0,
 '수고하세요': 0,
 '잘 부탁드립니다': 0,
 '고맙습니다': 0,
 '네네': 0,
 '예': 1224}

중복 문장/정보 빈도

In [14]:
def eda_repeated_lines(df, text_col='dialogue'):
    n_repeated = 0
    for text in df[text_col]:
        lines = [line.strip() for line in str(text).split('\n') if line.strip()]
        if len(lines) != len(set(lines)):
            n_repeated += 1
    print(f"=== [REPEATED LINES in {text_col}] ===")
    print(f"중복 문장이 포함된 샘플: {n_repeated} / {len(df)} ({100*n_repeated/len(df):.2f}%)")
    return n_repeated

In [15]:
eda_repeated_lines(train_df, 'dialogue')
eda_repeated_lines(train_df, 'summary')

=== [REPEATED LINES in dialogue] ===
중복 문장이 포함된 샘플: 55 / 12457 (0.44%)
=== [REPEATED LINES in summary] ===
중복 문장이 포함된 샘플: 0 / 12457 (0.00%)


0

In [22]:
def extract_masking_tokens(df, cols=['dialogue', 'summary']):
    token_pattern = r'#([^#\s]+)#'
    all_tokens = set()
    for col in cols:
        if col in df.columns:
            tokens_in_col = df[col].astype(str).apply(lambda x: re.findall(token_pattern, x))
            for token_list in tokens_in_col:
                for t in token_list:
                    all_tokens.add(f"#{t}#")
    print("=== [데이터 전체에서 자동 탐지된 마스킹 토큰 종류] ===")
    for t in sorted(all_tokens):
        print(t)
    print("총 개수:", len(all_tokens))
    return sorted(all_tokens)

In [23]:
mask_tokens_found = extract_masking_tokens(train_df)

=== [데이터 전체에서 자동 탐지된 마스킹 토큰 종류] ===
#Address#
#Alex#
#Bob#
#CarNumber#
#CardNumber#
#DateOfBirth#
#Email#
#FlightNumber#
#Kristin#
#Liliana#
#Mike#
#Name#
#PassportNumber#
#Person1#
#Person2#
#Person3#
#Person4#
#Person5#
#Person6#
#Person7#
#PersonName#
#PhoneNumber#
#Price#
#SSN#
총 개수: 24


In [24]:
mask_tokens_found = extract_masking_tokens(val_df)

=== [데이터 전체에서 자동 탐지된 마스킹 토큰 종류] ===
#Address#
#DateOfBirth#
#Person1#
#Person2#
#Person3#
#Person4#
#PhoneNumber#
총 개수: 7


In [21]:
mask_tokens_found = extract_masking_tokens(test_df)

KeyError: 'summary'